# 03 - Silver Layer

**Objetivo:** Criar a camada Silver da arquitetura Medallion com modelo dimensional (Star Schema).

Nesta etapa os dados da camada Bronze são transformados em um modelo dimensional bem estruturado, separando informações de negócio em tabelas de **dimensão** e **fato**, seguindo as melhores práticas de Data Warehousing.

### O que é feito nesta camada:
- Leitura dos dados limpos da camada Bronze (`credit_data_bronze.parquet`)
- Criação das tabelas de dimensão:
  - `dim_tempo`
  - `dim_produto`
  - `dim_consumidor`
- Criação da tabela fato (`fact_credito`) com métricas de negócio
- Geração de chaves surrogate (`sk_`) para cada dimensão
- Relacionamento entre fato e dimensões (Star Schema)
- Salvamento de cada tabela em formato Parquet na pasta `data/silver/`

**Importante:** Nesta camada são realizadas as transformações de modelagem dimensional.  
Não são feitas agregações avançadas nem cálculos complexos — o foco é construir um modelo limpo, performático e pronto para consumo no Power BI.

**Próxima camada:** [04_gold_layer.ipynb](./04_gold_layer.ipynb) → Agregações finais e camada otimizada para BI

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# ============================
# 03 - SILVER LAYER
# ============================

# ==================== CONFIGURAÇÃO ====================
bronze_path = Path('../data/bronze/credit_data_bronze.parquet')
silver_dir = Path('../data/silver')
silver_dir.mkdir(parents=True, exist_ok=True)

# ==================== 1. CARREGAR BRONZE ====================
df_bronze = pd.read_parquet(bronze_path)
print(f"Bronze carregada: {len(df_bronze):,} linhas")

Bronze carregada: 105,651 linhas


In [5]:
# ==================== 2. CRIAR DIMENSÕES ====================

# 2.1 dim_tempo
dim_tempo = pd.DataFrame({
    'sk_tempo': range(1, len(df_bronze['data_registro'].unique()) + 1),
    'data_registro': sorted(df_bronze['data_registro'].unique())
})
dim_tempo['ano'] = dim_tempo['data_registro'].dt.year
dim_tempo['mes'] = dim_tempo['data_registro'].dt.month
dim_tempo['dia'] = dim_tempo['data_registro'].dt.day

# Nomes em Português (Brasil)
dim_tempo['nome_mes'] = dim_tempo['data_registro'].dt.strftime('%B').map({
    'January': 'Janeiro', 'February': 'Fevereiro', 'March': 'Março',
    'April': 'Abril', 'May': 'Maio', 'June': 'Junho',
    'July': 'Julho', 'August': 'Agosto', 'September': 'Setembro',
    'October': 'Outubro', 'November': 'Novembro', 'December': 'Dezembro'
})

dim_tempo['dia_semana'] = dim_tempo['data_registro'].dt.strftime('%A').map({
    'Monday': 'Segunda-feira', 'Tuesday': 'Terça-feira', 'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira', 'Friday': 'Sexta-feira',
    'Saturday': 'Sábado', 'Sunday': 'Domingo'
})

In [7]:
# 2.2 dim_produto
dim_produto = (df_bronze[['tipo_credito']]
               .drop_duplicates()
               .reset_index(drop=True))
dim_produto['sk_produto'] = range(1, len(dim_produto) + 1)

In [9]:
# 2.3 dim_consumidor (perfil do cliente)
dim_consumidor = (df_bronze[['faixa_score_credito', 'faixa_renda', 
                             'flag_negativado', 'faixa_etaria']]
                  .drop_duplicates()
                  .reset_index(drop=True))
dim_consumidor['sk_consumidor'] = range(1, len(dim_consumidor) + 1)

In [11]:
# ==================== 3. CRIAR TABELA FATO ====================

fact_credito = df_bronze.copy()

# Fazer os joins para trazer as surrogate keys
fact_credito = fact_credito.merge(dim_tempo[['data_registro', 'sk_tempo']], 
                                  on='data_registro', how='left')

fact_credito = fact_credito.merge(dim_produto[['tipo_credito', 'sk_produto']], 
                                  on='tipo_credito', how='left')

fact_credito = fact_credito.merge(dim_consumidor[['faixa_score_credito', 'faixa_renda',
                                                  'flag_negativado', 'faixa_etaria', 'sk_consumidor']], 
                                  on=['faixa_score_credito', 'faixa_renda', 
                                      'flag_negativado', 'faixa_etaria'], 
                                  how='left')

# Selecionar apenas as colunas necessárias para a Fact
fact_credito = fact_credito[[
    'sk_tempo',
    'sk_produto',
    'sk_consumidor',
    'total_usuarios_simulando',
    'usuarios_elegiveis',
    'possui_oferta',
    'conversao_efetiva',
    'receita_gerada'
]]

In [12]:
fact_credito

,sk_tempo,sk_produto,sk_consumidor,total_usuarios_simulando,usuarios_elegiveis,possui_oferta,conversao_efetiva,receita_gerada
0,3,1,1,490,360.0,359.0,17.0,19.5
1,3,2,1,490,20.0,11.0,5.0,6.0
2,3,3,1,490,66.0,43.0,10.0,0.0
3,3,4,1,490,2.0,0.0,0.0,0.0
4,14,2,2,7196,40.0,2.0,0.0,0.0
...,...,...,...,...,...,...,...,...
105646,5,1,1198,1,2.0,2.0,0.0,0.0
105647,5,3,1198,1,1.0,1.0,0.0,0.0
105648,2,5,102,2,2.0,0.0,0.0,0.0
105649,16,5,795,2,1.0,0.0,0.0,0.0


In [13]:

# ==================== 4. SALVAR EM SILVER ====================

dim_tempo.to_parquet(silver_dir / 'dim_tempo.parquet', index=False)
dim_produto.to_parquet(silver_dir / 'dim_produto.parquet', index=False)
dim_consumidor.to_parquet(silver_dir / 'dim_consumidor.parquet', index=False)
fact_credito.to_parquet(silver_dir / 'fact_credito.parquet', index=False)

print(" Camada Silver criada com sucesso!")
print(f"   • dim_tempo:       {len(dim_tempo):,} linhas")
print(f"   • dim_produto:     {len(dim_produto):,} linhas")
print(f"   • dim_consumidor:  {len(dim_consumidor):,} linhas")
print(f"   • fact_credito:    {len(fact_credito):,} linhas")

 Camada Silver criada com sucesso!
   • dim_tempo:       28 linhas
   • dim_produto:     7 linhas
   • dim_consumidor:  1,315 linhas
   • fact_credito:    105,651 linhas
